# Lecture 3: Lean Replay and Axiom Audit

The verified repositories already contain Lean artifacts. PACTA does not run Charon, Aeneas, extraction, or Rust-to-Lean regeneration. It treats shipped Lean files as the verification artifact and focuses on replaying and inspecting them.


## Learning Objectives

- Explain the difference between transpilation and proof replay.
- Understand how PACTA discovers Lean files and manifests.
- Build a portable Lean invocation without Linux-only shell helpers.
- Explain `#print axioms` and why axiom sets matter.
- Diagnose missing Lean/lake or missing pinned Aeneas environments.


## Why Not Run Charon/Aeneas?

The corpus policy is strict:

- The transpilation work is finished in the verified repos.
- Re-running extraction could create a different artifact and confuse the trust story.
- The current task is to interpret, replay, summarize, and score the existing Lean proof artifacts.

For a production assurance case, translation faithfulness remains part of the trusted base unless separately proven.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "src" / "pacta").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from pacta.manifest import discover_layout

fixture = repo_root / "tests" / "fixtures" / "mini-ed25519-verified"
layout = discover_layout(fixture, "verification")
print("verification_dir:", layout.verification_dir)
print("files:")
for path in layout.compile_order:
    print(" ", path.relative_to(fixture))
print("warnings:", layout.warnings)


## Portable Lean Invocation

PACTA avoids repository `check.sh` scripts because those may assume Linux-only tools like `free`, `taskset`, or GNU `timeout`. Instead it uses Python `subprocess.run(..., timeout=...)` and constructs a Lean environment where `verification/gen` and `verification` are visible through `LEAN_PATH`.

If a pinned Aeneas Lean project is needed, PACTA can source a configured environment script and run `lake env lean`. It still does not run extraction.


In [ ]:
from pacta.lean import LeanTools, build_lean_invocation

tools = LeanTools(lean="/usr/bin/lean", lake="/usr/bin/lake")
example_file = layout.compile_order[0]
print(build_lean_invocation(example_file, tools, use_lake_env=True, output_path=example_file.with_suffix(".olean")))


## Axiom Audit

A theorem can compile while depending on unexpected axioms. For the Ed25519 arithmetic profiles, the expected axiom set is usually:

- `propext`
- `Classical.choice`
- `Quot.sound`

PACTA generates a temporary Lean file with imports such as:

```lean
import Proofs.FieldMain
import Proofs.EdMain
#print axioms CurveFieldProofs.fieldImplementation
#print axioms CurveFieldProofs.edwardsImplementation
```

It then parses Lean output and marks the result clean only when observed axioms match the expected set.


In [ ]:
from pacta.lean import parse_axiom_output

output = """'CurveFieldProofs.fieldImplementation' depends on axioms:
[propext, Classical.choice, Quot.sound]
'CurveFieldProofs.edwardsImplementation' depends on axioms:
[propext, Classical.choice, Quot.sound]
"""
parsed = parse_axiom_output(
    output,
    [
        "CurveFieldProofs.fieldImplementation",
        "CurveFieldProofs.edwardsImplementation",
    ],
)
print(parsed)


## Local Replay Failure Modes

Important distinctions:

- Missing `lean`: local verifier capability unavailable.
- Missing `lake`: local project environment may be unavailable.
- Missing Aeneas Lean project: local replay unavailable for repos that depend on it.
- Lean file fails: proof replay failed in this environment.
- Axiom set dirty: theorem depends on unexpected assumptions.

These are not the same. A professional report must state which one happened.


## Exercises

- Run `pacta doctor --config examples/repos.yaml --repo-name dalek-ed25519-verified` and classify the result.
- Create a fake axiom output with an extra axiom. Parse it and explain why the result should be dirty.
- Explain why a replay runner should not silently fall back from failure to an offline fixture.
